In [ ]:
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display

import hierarchical_position_classifier as hpc_module

importlib.reload(hpc_module)

from hierarchical_position_classifier import (
    combine_position_experiment_summaries,
    per_room_hierarchical_summary,
    run_global_position_experiment,
    run_global_position_experiments_by_split,
    run_global_position_experiments_by_split_knn,
    run_hierarchical_position_experiment,
    run_hierarchical_position_experiments_by_split,
    run_hierarchical_position_experiments_by_split_knn,
    summarize_distance_errors,
)
from utils.csi_preprocessing import process_magnitude_data
from utils.feature_pipeline import build_frequency_feature_dataframes
import utils.graphs as graphs

importlib.reload(graphs)

from utils.graphs import (
    plot_band_error_boxplot,
    plot_band_error_cdf,
    plot_floor_plan_heatmap,
    plot_global_position_confusion_matrix,
    plot_localization_error_cdf_by_model,
    plot_position_confusion_by_true_room,
    plot_position_confusion_when_room_correct,
)
from utils.import_data import get_csv_files, sort_meta_info
from utils.thesis_csv_processing import process_csv_files
from utils.cache import get_all_dataframes, get_results_path, save_summary, write_manifest


#### Configuration

In [ ]:
PROJECT_ROOT = Path(r"C:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project")
DATA_DIR = PROJECT_ROOT / "CSI DATA"
CALIBRATION_MODE = "rssi"

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
    "calibration_eps": 1e-12,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "apply_agc_compensation": False,
    "agc_reference": "median",
    "filter_method": "none",
    "filter_window": 5,
    "normalization": "none",
    "epsilon": 1e-8,
}

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": 60,
    "overlap_size": 30,
    "calibrate": False,
    "require_all_esps": False,
}

# Feature columns are stored as float32 to reduce memory footprint.
# Clear .cache/dataframes/ before running if you have older
# float64 caches from a previous run.

# ESP-to-room assignment for the local-ESP hierarchical classifier.
ROOM_LOCAL_ESPS = {
    1: (
        "esp_06", "esp_07", "esp_08", "esp_09", "esp_10",
        "esp_16", "esp_17", "esp_18", "esp_19", "esp_20",
    ),
    2: ("esp_01", "esp_02", "esp_03", "esp_11", "esp_12", "esp_13"),
    3: ("esp_04", "esp_05", "esp_14", "esp_15"),
}

# Experiment parameters.
SPLIT_MODES = ("random", "block")
BLOCK_COUNT = 10
TEST_SIZE = 0.30
TUNING_FORCE_RECOMPUTE = False  # set True to force a fresh direct grid search
RANDOM_STATE = 42
ROW_SPACING = 1.0
COLUMN_SPACING = 1.0

# Dataset shown in confusion matrix and floor plan cells.
CONFUSION_DATASET = "Fusion"   # "2.4 GHz", "5 GHz", or "Fusion"

# Toggle plot sections.
SHOW_CDF_PLOTS = True
SHOW_BOXPLOT = True
SHOW_FLOOR_PLAN = True
SHOW_CONFUSION_MATRICES = True
SHOW_PER_ROOM_PLOTS = False

# ── Derived cache / results paths ────────────────────────────────────────────
preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)

results_dir = get_results_path(preproc_opts, feat_opts)
plots_dir = results_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)


def _slugify(s: str) -> str:
    return (
        s.lower()
        .replace(".", "-")
        .replace(" ", "-")
        .replace("(", "")
        .replace(")", "")
        .strip("-")
    )


#### Data Loading and Preprocessing

In [ ]:
all_data_files = get_csv_files(str(DATA_DIR))
scenarios_id, locations_id, users_id, esps_id, trials_id = sort_meta_info(str(DATA_DIR))
print(f"Scenarios present: {', '.join(scenarios_id) or 'none'}")
print(f"Locations: {len(locations_id)}  |  Users: {len(users_id)}  |  ESPs: {len(esps_id)}")

##### Magnitude Data

In [ ]:
magnitude_data, agc_gain_data, csv_diagnostics = process_csv_files(
    all_data_files,
    return_diagnostics=True,
    calibration_mode=CALIBRATION_MODE,
    **CSV_PROCESSING_OPTIONS,
)

##### Segment data, extract features, and creat 3 dataframes

In [ ]:
def _build_feature_dataframes():
    processed, _ = process_magnitude_data(magnitude_data, agc_gain_data, **preproc_opts)
    return build_frequency_feature_dataframes(processed, **feat_opts)


feature_dataframes = get_all_dataframes(preproc_opts, feat_opts, _build_feature_dataframes)
df_24ghz = feature_dataframes["2.4 GHz"]
df_5ghz = feature_dataframes["5 GHz"]
df_fusion = feature_dataframes["Fusion"]

for name, df in feature_dataframes.items():
    print(f"{name}: {df.shape[0]} windows, {df.shape[1]} columns")

#### Hyperparameter Tuning

In [ ]:
""" import csv

TUNED_SUMMARY_PATH = results_dir / "tuning" / "tuned_summary.csv"

if not TUNED_SUMMARY_PATH.exists():
    msg = (
        f"Tuned summary not found at {TUNED_SUMMARY_PATH}. "
        "Run the Hyperparameter Tuning section first."
    )
    raise FileNotFoundError(msg)


def _optional_int(value, default=None):
    if value in (None, "", "None"):
        return default
    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass
    return int(float(value))


def _params_from_tuned_row(row: dict) -> dict:
    return {
        "n_estimators": int(float(row["n_estimators"])),
        "max_features": str(row["max_features"]),
        "max_depth": _optional_int(row.get("max_depth"), default=None),
        "min_samples_split": _optional_int(row.get("min_samples_split"), default=2),
        "min_samples_leaf": int(float(row["min_samples_leaf"])),
    }


tuned_params_lookup: dict[tuple[str, str], dict] = {}
with open(TUNED_SUMMARY_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        key = (row["dataset"], row["classifier"])
        tuned_params_lookup[key] = _params_from_tuned_row(row) """

tuned_params_lookup: dict[tuple[str, str], dict] = {
    ("2.4 GHz", "Global"): {
        "n_estimators": 500,
        "max_features": "log2",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    ("2.4 GHz", "Hierarchical"): {
        "n_estimators": 500,
        "max_features": "sqrt",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    ("5 GHz", "Global"): {
        "n_estimators": 500,
        "max_features": "sqrt",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    ("5 GHz", "Hierarchical"): {
        "n_estimators": 500,
        "max_features": "sqrt",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    ("Fusion", "Global"): {
        "n_estimators": 500,
        "max_features": "log2",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
    ("Fusion", "Hierarchical"): {
        "n_estimators": 500,
        "max_features": "log2",
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
    },
}

print("Loaded tuned hyperparameters (hardcoded from previous run):")
for k, v in tuned_params_lookup.items():
    print(f"  {k}: {v}")


#### Global 52-position Classifier

##### Perform {RF, KNN, SVM} for {2.4 GHz, 5 GHz, Fusion}

In [ ]:
global_predictions_by_split: dict[tuple[str, str], pd.DataFrame] = {}
global_models_by_split: dict[tuple[str, str], object] = {}
global_summary_rows = []

for band, dataframe in feature_dataframes.items():
    params = tuned_params_lookup[(band, "Global")]
    for split_mode in SPLIT_MODES:
        model, predictions, metrics = run_global_position_experiment(
            dataframe,
            dataset_name=band,
            split_mode=split_mode,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            row_spacing=ROW_SPACING,
            column_spacing=COLUMN_SPACING,
            n_blocks=BLOCK_COUNT,
            **params,
        )
        global_predictions_by_split[(band, split_mode)] = predictions
        global_models_by_split[(band, split_mode)] = model
        global_summary_rows.append(
            {
                "dataset": band,
                "split": split_mode,
                "n_estimators": params["n_estimators"],
                "max_features": params["max_features"],
                "max_depth": params["max_depth"],
                "min_samples_split": params["min_samples_split"],
                "min_samples_leaf": params["min_samples_leaf"],
                **metrics,
            }
        )

global_preds_by_split = global_predictions_by_split
global_summary = pd.DataFrame(global_summary_rows)
display(global_summary)
import gc
gc.collect()


### Result Analysis

#### Localization Error

#### CDF - distance error

#### Box plot - distance error

### Floor Plan Heatmap

### Confusion Matrices